In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
import joblib
import os

In [4]:
RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
MODELS_DIR = "models"
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

In [5]:
COLUMN_MAP = {
    "Amount Requested": "loan_amnt",
    "Debt-To-Income Ratio": "dti",
    "Employment Length": "emp_length",
    "Zip Code": "zip_code",
    "State": "addr_state",
}

In [6]:
print("Loading raw CSVs...")
accepted = pd.read_csv(os.path.join(RAW_DIR, "/Users/shreya/Desktop/Loan Predictor/data/accepted_2007_to_2018Q4.csv"), low_memory=False)
rejected = pd.read_csv(os.path.join(RAW_DIR, "/Users/shreya/Desktop/Loan Predictor/data/rejected_2007_to_2018Q4.csv"), low_memory=False)
rejected = rejected.rename(columns=COLUMN_MAP)


Loading raw CSVs...


In [7]:
print("Accepted shape:", accepted.shape)
print("Rejected shape:", rejected.shape)

Accepted shape: (2260701, 151)
Rejected shape: (27648741, 9)


In [8]:
print("\nBuilding approval dataset...")
SHARED_COLS = ["loan_amnt", "dti", "emp_length", "zip_code", "addr_state"]



Building approval dataset...


In [9]:
acc_common = accepted[SHARED_COLS].copy()
acc_common["approved"] = 1

rej_common = rejected[SHARED_COLS].copy()
rej_common["approved"] = 0


In [10]:
for df_ in (acc_common, rej_common):
    if df_["dti"].dtype == object:
        df_["dti"] = df_["dti"].astype(str).str.replace("%", "", regex=False)
        df_["dti"] = pd.to_numeric(df_["dti"], errors="coerce")

approval_df = pd.concat([acc_common, rej_common], ignore_index=True).dropna()


In [11]:
approved_rows = approval_df[approval_df["approved"] == 1]
rejected_rows = approval_df[approval_df["approved"] == 0]
rejected_sample = resample(rejected_rows, n_samples=len(approved_rows), random_state=42)
approval_balanced = pd.concat([approved_rows, rejected_sample]).reset_index(drop=True)


In [18]:
approval_encoders = {}

for col in ["emp_length", "addr_state", "zip_code"]:
    le = LabelEncoder()

    approval_balanced[col] = le.fit_transform(
        approval_balanced[col].astype(str)
    )

    approval_encoders[col] = le

In [19]:
joblib.dump(approval_encoders, os.path.join(MODELS_DIR, "approval_encoders.pkl"))

X_app = approval_balanced.drop(columns="approved")
y_app = approval_balanced["approved"]

X_train_app, X_test_app, y_train_app, y_test_app = train_test_split(
    X_app, y_app, test_size=0.2, stratify=y_app, random_state=42
)

X_train_app.assign(approved=y_train_app).to_csv(
    os.path.join(PROCESSED_DIR, "approval_train.csv"), index=False)
X_test_app.assign(approved=y_test_app).to_csv(
    os.path.join(PROCESSED_DIR, "approval_test.csv"), index=False)

print("Approval dataset saved. Train:", X_train_app.shape, "Test:", X_test_app.shape)


Approval dataset saved. Train: (3381835, 5) Test: (845459, 5)


In [20]:
print("\nBuilding default dataset...")
default_df = accepted[accepted["loan_status"].isin(
    ["Fully Paid", "Charged Off", "Default"])].copy()
default_df["target"] = default_df["loan_status"].apply(
    lambda x: 1 if x in ["Charged Off", "Default"] else 0)

missing_pct = default_df.isnull().mean()
default_df = default_df.drop(columns=missing_pct[missing_pct > 0.5].index)



Building default dataset...


In [21]:
drop_cols = [c for c in ["id", "member_id", "url", "desc", "emp_title", "loan_status"]
             if c in default_df.columns]
default_df = default_df.drop(columns=drop_cols)

for col in default_df.select_dtypes(include="number").columns:
    default_df[col] = default_df[col].fillna(default_df[col].median())
for col in default_df.select_dtypes(include="object").columns:
    default_df[col] = default_df[col].fillna(default_df[col].mode()[0])


In [22]:
default_encoders = {}
categorical_cols = default_df.select_dtypes(include="object").columns.tolist()
for col in categorical_cols:
    le = LabelEncoder()
    default_df[col] = le.fit_transform(default_df[col].astype(str))
    default_encoders[col] = le

joblib.dump(default_encoders, os.path.join(MODELS_DIR, "default_encoders.pkl"))

numeric_cols = default_df.select_dtypes(include="number").columns.drop("target")
scaler = StandardScaler()
default_df[numeric_cols] = scaler.fit_transform(default_df[numeric_cols])
joblib.dump(scaler, os.path.join(MODELS_DIR, "default_scaler.pkl"))

X_def = default_df.drop(columns="target")
y_def = default_df["target"]

X_train_def, X_test_def, y_train_def, y_test_def = train_test_split(
    X_def, y_def, test_size=0.2, stratify=y_def, random_state=42
)


In [17]:
print("\nBefore SMOTE:")
print(y_train_def.value_counts())

MAX_TRAIN_SIZE = 50000

if len(X_train_def) > MAX_TRAIN_SIZE:

    X_train_sample, _, y_train_sample, _ = train_test_split(
        X_train_def,
        y_train_def,
        train_size=MAX_TRAIN_SIZE,
        stratify=y_train_def,
        random_state=42
    )

else:
    X_train_sample = X_train_def
    y_train_sample = y_train_def

print("Training data used for SMOTE:", X_train_sample.shape)

# Apply SMOTE
smote = SMOTE(random_state=42)

X_train_def_res, y_train_def_res = smote.fit_resample(
    X_train_sample,
    y_train_sample
)

print("\nAfter SMOTE:")
print(pd.Series(y_train_def_res).value_counts())

# Save balanced training data
X_train_def_res.assign(
    target=y_train_def_res
).to_csv(
    os.path.join(PROCESSED_DIR, "default_train.csv"),
    index=False
)

# Save original test data WITHOUT SMOTE
X_test_def.assign(
    target=y_test_def
).to_csv(
    os.path.join(PROCESSED_DIR, "default_test.csv"),
    index=False
)

print(
    "\nDefault dataset saved.",
    "Train after SMOTE:", X_train_def_res.shape,
    "Test:", X_test_def.shape
)

print("\nDone. Processed files are in data/processed/, encoders/scaler in models/.")


Before SMOTE:
target
0    861401
1    214879
Name: count, dtype: int64
Training data used for SMOTE: (50000, 89)

After SMOTE:
target
0    40018
1    40018
Name: count, dtype: int64

Default dataset saved. Train after SMOTE: (80036, 89) Test: (269070, 89)

Done. Processed files are in data/processed/, encoders/scaler in models/.
